# Zero Trust Authentication Based on ROS2
本專題以公開的 [ROSpace: Intrusion Detection Dataset for a ROS2-Based Cyber-Physical System](https://arxiv.org/abs/2402.08468) 為基礎，並參考其作者提供的程式碼庫 [rospace_dataset](https://github.com/TommasoPuccetti/rospace_dataset) 。我們的目標是復刻並簡化原始流程，建立一個可操作的版本，透過資料集訓練異常檢測模型，進一步驗證 Zero Trust 架構在 ROS2 環境下的可行性。

> This project is based on the publicly available [ROSpace: Intrusion Detection Dataset for a ROS2-Based Cyber-Physical System](https://arxiv.org/abs/2402.08468) and references the code repository [rospace_dataset](https://github.com/TommasoPuccetti/rospace_dataset) provided by its authors. Our goal is to replicate and simplify the original process, build a workable version, train an anomaly detection model using the dataset, and further verify the feasibility of the Zero Trust architecture in a ROS2 environment.

整個流程我們將其拆分成 Data Collection、Data Preprcessing、Model Training、Application 做簡單的區分。

### 1. Data Collection
架構可以參考 [Simple Architecture Image](README/architecture.png)，主要的攻擊流程：

##### (1) NMAP Discovery / Port Scanning

##### (2) ROS2 Reconnaissance

##### (3) NMAP SYN Flood (IPv6 RA Flood)

##### (4) ROS2 Reflection

##### (5) ROS2 Node Crashing

##### (6) Metasplot SYN Flood


### 2. Data Preprocessing

In [4]:
import os

# 1. 讓使用者輸入日期
date_val = input("請輸入要來前處理的日期資料 (例如 0507, 0509): ")
print(f"準備處理 {date_val} 的資料...")

# 2. 定位專案根目錄
current_dir = os.getcwd()
project_name = "Zero-Trust-Authentication-Based-on-ROS2"
if project_name in current_dir:
    # 切割路徑，抓到根目錄
    root_dir = current_dir[:current_dir.find(project_name) + len(project_name)]
else:
    root_dir = current_dir

processing_dir = os.path.join(root_dir, "rospace_dataset", "1_processing")

# 3. 動態組合所有的檔案絕對路徑
pcapng_path = os.path.join(root_dir, "raspberry_subscriber", f"monitor_{date_val}", "Tshark", "temp_capture.pcapng")
os_path = os.path.join(root_dir, "raspberry_subscriber", f"monitor_{date_val}", "OS", "OS_monitor.csv")
ros_path = os.path.join(root_dir, "raspberry_subscriber", f"monitor_{date_val}", "ROSBags", "ros2_monitor.csv")
attack_path = os.path.join(root_dir, "rospace_dataset", "attack", f"attacks_log_{date_val}.csv")

converted_net_path = os.path.join(processing_dir, "converted", "temp_capture.csv")

# 4. 預先檢查 pcapng 檔案存不存在
if not os.path.exists(pcapng_path):
    raise FileNotFoundError(f"找不到封包檔案，請確認日期是否輸入正確或檔案是否存在：\n{pcapng_path}")

print("所有檔案路徑組合且確認完畢！可以執行下一步。")

準備處理 0509 的資料...
所有檔案路徑組合且確認完畢！可以執行下一步。


In [5]:
import subprocess

print("開始將 pcapng 轉換為 CSV...")

try:
    # cwd 參數代表「要在哪個資料夾下執行這行指令」，這樣就不用 %cd 了
    # check=True 代表只要指令失敗，就會立刻觸發 CalledProcessError
    subprocess.run(
        ["python", "custom_pcapng2csv.py", pcapng_path],
        cwd=processing_dir,
        check=True
    )
    print("網路封包轉換完成！")
    
except subprocess.CalledProcessError:
    print("轉換過程發生錯誤，程式已停止執行！")

開始將 pcapng 轉換為 CSV...
網路封包轉換完成！


In [6]:
import subprocess

print("開始進行四方資料合併...")

try:
    subprocess.run(
        ["python", "csv_merging_parallel.py", "-s", os_path, "-n", converted_net_path, "-r", ros_path, "-a", attack_path],
        cwd=processing_dir,
        check=True
    )
    print("前處理大功告成！請去 merged-dataset 資料夾領取你的新鮮資料集。")
    
except subprocess.CalledProcessError:
    print("合併過程發生錯誤，程式已停止執行！")

開始進行四方資料合併...
前處理大功告成！請去 merged-dataset 資料夾領取你的新鮮資料集。


### 3. Model Training

### 4. Application